# Interferometric Coherence and Closure Phase per ISMN Station

<br>

For each ISMN soil moisture station, this notebook reads the per-acquisition GSLC
window chips saved by
[`03_get_NISAR_GSLC_windows.ipynb`](03_get_NISAR_GSLC_windows.ipynb)
under `nisar_data/spain_GSLC_windows/` and computes, over a small window centered
on the station's pixel:

- **pairwise coherence** for every sequential (12-day, consecutive-in-time) pair of
  acquisitions from the same relative orbit/track
- **closure phase** for every sequential (24-day, sliding/overlapping) triplet from
  the same track

One CSV per station is written for each: `{station_name}_NISAR_GSLC_coherence.csv`
and `{station_name}_NISAR_GSLC_closure_phase.csv`, under
`nisar_data/spain_GSLC_coherence_closure/`.

This is purely local, in-memory processing over already-downloaded window chips —
no network access, no `earthaccess` login.

:::{note} Method
Implemented by `process_station()` in
[`util/nisar_coherence_closure.py`](../util/nisar_coherence_closure.py) — the same
shared module used by
[`04_calc_coh_phase_closure_NISAR_austria.ipynb`](04_calc_coh_phase_closure_NISAR_austria.ipynb).

- **Grouping**: saved windows for a station are grouped by relative orbit/track
  (parsed from each file's name/attrs) and sorted by acquisition time within each
  track. Sequential pairs are `(i, i+1)`; sequential triplets are the overlapping
  sliding window `(i, i+1, i+2)` — e.g. for four acquisitions A/B/C/D: pairs
  AB, BC, CD and triplets ABC, BCD.
- **Estimation window**: a 3x3 window centered on the station's pixel (the same
  center used throughout this cookbook's GCOV/GSLC pipelines), with pixels flagged
  invalid by the saved `mask` layer (values 0/255, the same convention used
  elsewhere in this cookbook) excluded from the window sums.
- **Coherence**: the standard sample estimator
  `|sum(s1 * conj(s2))| / sqrt(sum(|s1|^2) * sum(|s2|^2))` over the window's valid
  pixels.
- **Closure phase**: `angle(z12 * z23 * conj(z13))` where `z_ij = sum(s_i * conj(s_j))`
  over the window — equivalent to wrapping `phase(z12) + phase(z23) - phase(z13)` to
  `(-pi, pi]`. Uses the pixels valid across all three acquisitions (not just each
  pairwise-valid set), so the same pixel set contributes to every leg of the loop.
- **No triplets are dropped.** Every sequential triplet gets a closure-phase row,
  regardless of how low its coherence is. Each row also carries
  `min_coherence_HH`/`min_coherence_HV` — the minimum of that triplet's three
  pairwise coherences (legs 12, 23, 13) for that polarization — as a quality metric,
  so low-coherence triplets can be filtered out downstream (e.g.
  `min_coherence_HH >= 0.3`) without recomputing anything.
- **HH and HV** are computed independently and reported as side-by-side columns in
  the same CSV (`coherence_HH`/`coherence_HV`, `closure_phase_HH`/`closure_phase_HV`).
- **`local_incidence_angle_deg`** for a row is the mean of the participating
  acquisitions' own center-pixel `local_incidence_angle_deg` (already computed and
  saved per-acquisition by the GSLC window notebook) — 2 acquisitions averaged for
  a pair row, 3 for a triplet row.
- Both CSVs are fully rebuilt from scratch on every run of this notebook (cheap,
  since there's no network cost involved).
:::

<hr>

## Overview

1. Load station list
2. Process each station: group by track, compute pairwise coherence and
   triplet closure phase, save CSVs
3. (Optional) Preview one station's output

<hr>

In [1]:
import sys
import os
import json
from pathlib import Path

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from util.nisar_coherence_closure import process_station

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Load station list

Same station list as the earlier notebooks in this cookbook. Only the names are
needed here — the pairwise coherence/closure-phase computation reads everything else
(geometry, mask, complex SLC values) straight out of each station's saved `.nc`
window files.

In [2]:
# This notebook was originally written for the Spain AOI; the actual target has
# since moved to Alaska (same fix already applied in 02_get_NISAR_data.ipynb
# and 03_get_NISAR_GSLC_windows.ipynb) -- ismn_data/spain/ doesn't exist,
# only ismn_data/alaska/ does.
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

station_names = [feat["properties"]["station_val"] for feat in stations_geojson["features"]]

print(f"Loaded {len(station_names)} station names")
station_names

Loaded 17 station names


['MooseInc',
 'AtigunPass',
 'Coldfoot',
 'EagleSummit',
 'GobblersKnob',
 'GraniteCrk',
 'ImnaviatCreek',
 'LittleChenaRidge',
 'McGrath',
 'NukaGlacier',
 'PilgrimHotSprings',
 'RockyPoint',
 'SpringCreek',
 'SusitnaValleyHigh',
 'TurnagainPass',
 'UpperNomeCreek',
 'Kenai-29-ENE']

## 2. Process each station: group by track, compute coherence and closure phase, save CSVs

For each station, `process_station()`:

1. Loads every saved GSLC window `.nc` file for that station from `gslc_dir`,
   grouped by relative orbit/track and sorted by acquisition time.
2. For each track, computes HH/HV coherence for every sequential pair, and HH/HV
   closure phase for every sequential triplet — all of them, none dropped — along
   with each triplet's `min_coherence_HH`/`min_coherence_HV` quality metric (see the
   note at the top of this notebook).
3. Saves `{station}_NISAR_GSLC_coherence.csv` and
   `{station}_NISAR_GSLC_closure_phase.csv` under `output_dir`.

A station with no saved window files (not yet downloaded, or none passed the
GSLC notebook's track filtering) ends up with two empty CSVs (header only) rather
than an error.

In [3]:
COHERENCE_WINDOW_HALF = 1  # window half-size for coherence/closure phase estimation; 1 = 3x3 (default), 2 = 5x5, etc.
# Larger reduces the small-sample bias in the coherence estimate (worst at low true coherence with the 3x3 default)
# at the cost of spatial resolution around the station.

# This notebook was originally written for the Spain AOI; the actual target has
# since moved to Alaska (same fix already applied in 02_get_NISAR_data.ipynb
# and 03_get_NISAR_GSLC_windows.ipynb) -- fixed here so this reads the real
# saved windows instead of an empty spain_GSLC_windows/ directory.
gslc_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"
output_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_coherence_closure"

for station_name in station_names:
    coherence_path, closure_path = process_station(
        station_name, gslc_dir, output_dir, window_half=COHERENCE_WINDOW_HALF
    )
    n_coherence = sum(1 for _ in open(coherence_path)) - 1
    n_closure = sum(1 for _ in open(closure_path)) - 1
    print(f"{station_name}: {n_coherence} pair(s) -> {coherence_path.name}, "
          f"{n_closure} triplet(s) -> {closure_path.name}")

MooseInc: 36 pair(s) -> MooseInc_NISAR_GSLC_coherence.csv, 34 triplet(s) -> MooseInc_NISAR_GSLC_closure_phase.csv
AtigunPass: 32 pair(s) -> AtigunPass_NISAR_GSLC_coherence.csv, 28 triplet(s) -> AtigunPass_NISAR_GSLC_closure_phase.csv
Coldfoot: 36 pair(s) -> Coldfoot_NISAR_GSLC_coherence.csv, 34 triplet(s) -> Coldfoot_NISAR_GSLC_closure_phase.csv
EagleSummit: 52 pair(s) -> EagleSummit_NISAR_GSLC_coherence.csv, 48 triplet(s) -> EagleSummit_NISAR_GSLC_closure_phase.csv
GobblersKnob: 36 pair(s) -> GobblersKnob_NISAR_GSLC_coherence.csv, 34 triplet(s) -> GobblersKnob_NISAR_GSLC_closure_phase.csv
GraniteCrk: 49 pair(s) -> GraniteCrk_NISAR_GSLC_coherence.csv, 45 triplet(s) -> GraniteCrk_NISAR_GSLC_closure_phase.csv
ImnaviatCreek: 28 pair(s) -> ImnaviatCreek_NISAR_GSLC_coherence.csv, 22 triplet(s) -> ImnaviatCreek_NISAR_GSLC_closure_phase.csv
LittleChenaRidge: 57 pair(s) -> LittleChenaRidge_NISAR_GSLC_coherence.csv, 58 triplet(s) -> LittleChenaRidge_NISAR_GSLC_closure_phase.csv
McGrath: 31 pair

## 3. (Optional) Preview one station's output

In [20]:
import pandas as pd

preview_station = station_names[0]
coherence_df = pd.read_csv(output_dir / f"{preview_station}_NISAR_GSLC_coherence.csv")
closure_df = pd.read_csv(output_dir / f"{preview_station}_NISAR_GSLC_closure_phase.csv")

print(f"--- {preview_station} coherence (pairs) ---")
display(coherence_df)
print(f"\n--- {preview_station} closure phase (triplets) ---")
display(closure_df)

--- MooseInc coherence (pairs) ---


,date_1,date_2,temporal_baseline_days,orbit_dir,orbit_num,local_incidence_angle_deg,coherence_HH,coherence_HV
0,2026-06-27T05:52:57,2026-07-09T05:52:56,11.999988,Descending,15,45.806715,0.530313,0.401262
1,2026-07-09T05:52:56,2026-07-21T05:52:56,12.000000,Descending,15,45.804586,0.560371,0.660659
2,2026-07-21T05:52:56,2026-08-14T05:52:54,23.999977,Descending,15,45.807825,0.284550,0.133732
3,2026-06-19T14:30:45,2026-07-01T14:30:44,11.999988,Ascending,78,37.931900,0.681133,0.694660
4,2026-07-01T14:30:44,2026-07-13T14:30:44,12.000000,Ascending,78,37.934389,0.324555,0.220750
5,2026-07-13T14:30:44,2026-07-25T14:30:43,11.999988,Ascending,78,37.933218,0.536790,0.222754
6,2026-07-25T14:30:43,2026-08-18T14:30:42,23.999988,Ascending,78,37.929852,0.609896,0.288202
7,2026-06-20T05:44:38,2026-07-02T05:44:37,11.999988,Descending,87,40.244844,0.490746,0.711807
8,2026-07-02T05:44:37,2026-07-14T05:44:37,12.000000,Descending,87,40.249632,0.637512,0.788600
9,2026-07-14T05:44:37,2026-07-26T05:44:36,11.999988,Descending,87,40.249105,0.526627,0.468777



--- MooseInc closure phase (triplets) ---


,date_1,date_2,date_3,temporal_baseline_12_days,temporal_baseline_23_days,temporal_baseline_13_days,orbit_dir,orbit_num,local_incidence_angle_deg,closure_phase_HH,closure_phase_HV,min_coherence_HH,min_coherence_HV
0,2026-06-27T05:52:57,2026-07-09T05:52:56,2026-07-21T05:52:56,11.999988,12.000000,23.999988,Descending,15,45.806085,0.819857,0.095006,0.530313,0.401262
1,2026-07-09T05:52:56,2026-07-21T05:52:56,2026-08-14T05:52:54,12.000000,23.999977,35.999977,Descending,15,45.806666,-0.676905,-0.628313,0.284550,0.133732
2,2026-06-19T14:30:45,2026-07-01T14:30:44,2026-07-13T14:30:44,11.999988,12.000000,23.999988,Ascending,78,37.932608,0.866670,0.706333,0.324555,0.140997
3,2026-07-01T14:30:44,2026-07-13T14:30:44,2026-07-25T14:30:43,12.000000,11.999988,23.999988,Ascending,78,37.933730,-2.196489,-1.555961,0.142823,0.220750
4,2026-07-13T14:30:44,2026-07-25T14:30:43,2026-08-18T14:30:42,11.999988,23.999988,35.999977,Ascending,78,37.931243,0.252119,1.601884,0.204406,0.222754
5,2026-06-20T05:44:38,2026-07-02T05:44:37,2026-07-14T05:44:37,11.999988,12.000000,23.999988,Descending,87,40.246975,-0.096604,0.106726,0.490746,0.711807
6,2026-07-02T05:44:37,2026-07-14T05:44:37,2026-07-26T05:44:36,12.000000,11.999988,23.999988,Descending,87,40.248746,-0.511642,0.223941,0.526627,0.468777
7,2026-07-14T05:44:37,2026-07-26T05:44:36,2026-08-19T05:44:35,11.999988,23.999988,35.999977,Descending,87,40.246328,-0.167218,0.842912,0.287005,0.468777
8,2026-06-24T14:22:26,2026-07-06T14:22:25,2026-07-18T14:22:21,11.999988,11.999954,23.999942,Ascending,150,43.667533,-0.837824,-0.120491,0.359864,0.294734
9,2026-07-06T14:22:25,2026-07-18T14:22:21,2026-08-11T14:22:20,11.999954,23.999988,35.999942,Ascending,150,43.669056,-0.235580,-0.817729,0.359864,0.294734


<hr>

## Summary

You now have, under `nisar_data/spain_GSLC_coherence_closure/`, two CSVs per station:

**`{station}_NISAR_GSLC_coherence.csv`** — one row per sequential (12-day) acquisition
pair from the same track, columns: `date_1`, `date_2`, `orbit_dir`, `orbit_num`,
`local_incidence_angle_deg`, `coherence_HH`, `coherence_HV`. Written for every
sequential pair unconditionally.

**`{station}_NISAR_GSLC_closure_phase.csv`** — one row per sequential (24-day,
overlapping) acquisition triplet from the same track — every triplet, none dropped —
columns: `date_1`, `date_2`, `date_3`, `orbit_dir`, `orbit_num`,
`local_incidence_angle_deg`, `closure_phase_HH`, `closure_phase_HV` (radians, in
`(-pi, pi]`), `min_coherence_HH`, `min_coherence_HV` (the minimum of that triplet's
three pairwise coherences for that polarization). Filter on `min_coherence_HH`/`_HV`
downstream to discard low-coherence triplets as needed — nothing here needs
recomputing to change that threshold.

Re-running this notebook after more GSLC windows are downloaded (Section 4 of
[`03_get_NISAR_GSLC_windows.ipynb`](03_get_NISAR_GSLC_windows.ipynb))
simply rebuilds both CSVs from whatever `.nc` files currently exist — safe to re-run
at any time.

<hr>